# 🪐 Automated Exam Solver (Solusi Soal Ujian Otomatis)

Sistem penyelesaian soal ujian otomatis berbasis **Jupyter Notebook** yang dirancang khusus untuk mahasiswa tingkat sarjana **Astronomi, Fisika, dan Matematika**.

### 🌟 Arsitektur Sistem:
1. **Input & Inventarisasi:** Menerima berkas PDF ujian (baik dokumen digital maupun hasil pemindaian/foto).
2. **OCR & Transkripsi:** Menggunakan OCR lokal gratis dan terbuka (**Tesseract OCR & PyMuPDF**) tanpa ketergantungan API berbayar.
3. **Penyelesai Independen (Solver):** **Google Gemini (`gemini-3.8-flash`)** untuk menurunkan rumus, menyusun langkah matematika yang dapat diaudit, dan menghasilkan sintaks LaTeX yang rapi.
4. **Verifikasi Independen (Verifier):** **OpenRouter (`nvidia/nemotron-3.5-super-vl:free`)** untuk meninjau kecocokan rumus, akurasi aljabar, dan analisis dimensi secara objektif.
5. **Pemeriksaan Deterministik:** Pustaka komputasi sains lokal (**SymPy**) dan basis data konstanta astronomi standar (IAU/CODATA) untuk verifikasi numerik tanpa menghabiskan kuota AI.
6. **Perakitan LaTeX & Kompilasi PDF:** Menjaga keutuhan template `main.tex` (preamble, typography, stix2, baskervald, tcolorbox) dan mengompilasi PDF menggunakan **Tectonic**.
7. **Ketahanan & Pemulihan (Persistence & Recovery):** Menyimpan status pekerjaan (*checkpoint*) secara atomik setelah setiap soal, sehingga dapat dilanjutkan (*resume*) kapan saja tanpa pemrosesan ulang.


## 1. Pemeriksaan Konfigurasi & Dependensi

Memeriksa ketersediaan pustaka Python, engine OCR lokal (Tesseract), compiler LaTeX (Tectonic/pdflatex), dan integritas berkas template `main.tex`.


In [ ]:
import os
import json
from pathlib import Path
from exam_solver.config import config, mask_key, BASE_DIR, JOBS_DIR, TEMPLATE_PATH

# Jalankan validasi diagnostik sistem
status = config.validate()
print("=" * 60)
print("DIAGNOSTIK SISTEM EXAM SOLVER")
print("=" * 60)
for k, v in status.items():
    print(f"  {k:35}: {v}")
print("=" * 60)

if not status["template_exists"]:
    raise FileNotFoundError("Berkas template main.tex tidak ditemukan di direktori utama!")
if not status["latex_engine_available"]:
    print("PERINGATAN: Engine LaTeX belum terdeteksi di PATH.")


## 2. Validasi Kredensial API & Model

Model yang digunakan:
- **Solver**: Google Gemini (`gemini-3.8-flash`) via `GEMINI_API_KEY`.
- **Verifier**: OpenRouter (`nvidia/nemotron-3.5-super-vl:free`) via `OPENROUTER_API_KEY`.

Kunci API dimuat secara aman dari berkas `.env` atau *environment variables* tanpa pernah ditampilkan secara terbuka di keluaran notebook.


In [ ]:
print("Status Kredensial:")
print(f"  Gemini API Key     : {mask_key(config.gemini_api_key)} (Model: {config.gemini_model})")
print(f"  OpenRouter API Key : {mask_key(config.openrouter_api_key)} (Model: {config.openrouter_model})")

if not config.gemini_api_key:
    print("\n[CATATAN]: GEMINI_API_KEY belum disetel di .env.")
    print("Salin .env.example menjadi .env lalu isi kunci API Anda.")
if not config.openrouter_api_key:
    print("[CATATAN]: OPENROUTER_API_KEY belum disetel di .env.")
    print("Verifikasi mandiri akan mencatat status UNVERIFIED jika kunci belum ada.")


## 3. Pemilihan Berkas PDF Soal Ujian

Pilih berkas PDF soal ujian yang akan diproses. Anda dapat menggunakan soal contoh di `sample_exams/sample_exam_astronomy.pdf` (atau `sample_exams/sample_scanned_exam.pdf` untuk menguji OCR pemindaian) atau mengarahkan ke berkas PDF Anda sendiri.


In [ ]:
from exam_solver.ocr import validate_pdf

# Tentukan berkas PDF yang ingin diproses
PDF_INPUT_PATH = "sample_exams/sample_exam_astronomy.pdf"

# Validasi berkas PDF
pdf_meta = validate_pdf(PDF_INPUT_PATH)
print("Informasi Berkas PDF:")
print(f"  Lokasi Berkas  : {PDF_INPUT_PATH}")
print(f"  Jumlah Halaman : {pdf_meta['page_count']}")
print(f"  Ukuran Berkas  : {pdf_meta['file_size_bytes']} bytes")


## 4. OCR & Transkripsi Soal

Mengekstrak teks dari PDF menggunakan PyMuPDF (untuk teks digital) atau Tesseract OCR lokal (jika dokumen berupa gambar hasil pemindaian/foto).
Hasil mentah disimpan di `raw_ocr.txt` dan transkripsi terstruktur disimpan di `transcription.txt`.


In [ ]:
from exam_solver.state import JobManager
from exam_solver.workflow import ExamWorkflow

# Inisialisasi JobManager untuk mengelola direktori dan status penyimpanan
job_mgr = JobManager(source_pdf=PDF_INPUT_PATH)
print(f"Job ID Aktif: {job_mgr.job_id}")
print(f"Direktori Pekerjaan: {job_mgr.job_dir}")

# Inisialisasi Workflow
workflow = ExamWorkflow(job_mgr)

# Jalankan ekstraksi OCR dan pembuatan transkripsi
questions = workflow.run_ocr_and_transcription(dpi=200)

print(f"\nBerhasil mendeteksi {len(questions)} soal:")
for q in questions:
    flag_str = f" [AMBIGU: {q.ambiguity_note}]" if q.ambiguity_note else ""
    print(f"  - [{q.q_id}] {q.title} (Hal. {q.page_number}) - Status: {q.status}{flag_str}")

print(f"\nBerkas transkripsi tersimpan di: {job_mgr.transcription_file}")


## 5. Peninjauan Ambiguitas & Konfirmasi Pengguna

Jika terdapat simbol yang rusak, bagian yang tidak terbaca, parameter yang hilang, atau rujukan ke gambar/grafik tertentu, sistem menandai soal tersebut dengan status `NEEDS_REVIEW`.
Pengguna dapat memberikan klarifikasi sebelum soal diselesaikan.


In [ ]:
# Periksa apakah ada soal yang membutuhkan klarifikasi
ambiguous_questions = [q for q in job_mgr.get_all_questions() if q.status == "NEEDS_REVIEW"]

if ambiguous_questions:
    print(f"Ditemukan {len(ambiguous_questions)} soal yang memerlukan peninjauan:")
    for q in ambiguous_questions:
        print(f"\nSoal {q.q_id} (Hal. {q.page_number}):")
        print(f"  Catatan Ambiguitas: {q.ambiguity_note}")
        print(f"  Cuplikan Teks     : {q.cleaned_text[:120]}...")
else:
    print("Seluruh soal memiliki teks yang jelas. Tidak ada ambiguitas yang menghambat.")


## 6. Penyelesaian Soal (Gemini) & Pemeriksaan Matematika

Setiap soal diselesaikan secara independen oleh Gemini (`gemini-3.8-flash`) dengan identifikasi parameter yang diketahui, penurunan rumus bertahap, dan pengecekan konsistensi dimensi. Pemeriksaan aritmatika numerik dilakukan secara lokal menggunakan SymPy.


In [ ]:
from exam_solver.state import QuestionStatus

print(f"Memproses penyelesaian dan verifikasi soal untuk Job: {job_mgr.job_id}")

try:
    processed_questions = workflow.solve_all_questions(interactive=False)
    for q in processed_questions:
        print(f"Soal {q.q_id}: Status Akhir = {q.status} (Percobaan Solusi: {q.solution_attempts_count})")
except Exception as e:
    print(f"Penanganan eksekusi: {str(e)}")
    print("Status pekerjaan tetap tersimpan secara atomik di state.json.")


## 7. Hasil Verifikasi Independen (OpenRouter)

Melihat evaluasi independen dari model verifikator (`nvidia/nemotron-3.5-super-vl:free`).
Batas maksimal solusi per soal adalah 2 kali percobaan (1 solusi awal + 1 revisi perbaikan jika verifikasi pertama gagal).


In [ ]:
for q in job_mgr.get_all_questions():
    print("=" * 60)
    print(f"{q.title} - Status: {q.status}")
    print(f"Jumlah Percobaan Solusi: {q.solution_attempts_count}")
    for idx, v in enumerate(q.verifications, 1):
        print(f"  Verifikasi {idx}: Status={v.get('status')} | Temuan: {v.get('findings')}")
    print("=" * 60)


## 8. Perakitan Dokumen LaTeX

Menyusun berkas `.tex` lengkap dengan menyisipkan solusi ke dalam template `main.tex`.
Preamble, paket (baskervald, stix2, tcolorbox, tikz), serta tata letak dokumen dipertahankan secara utuh.


In [ ]:
# Tentukan metadata ujian untuk header dokumen
exam_meta = {
    "exam_type_year": "UTS 2024",
    "course_name": "AS2001 - Astrofisika",
    "semester": "Semester Ganjil"
}

# Rakit dokumen LaTeX
tex_content = workflow.latex_engine.generate_latex(job_mgr.get_all_questions(), exam_metadata=exam_meta)
with open(job_mgr.tex_file, "w", encoding="utf-8") as f:
    f.write(tex_content)

print(f"Berkas LaTeX berhasil dirakit: {job_mgr.tex_file}")
print(f"Ukuran berkas .tex: {job_mgr.tex_file.stat().st_size} bytes")


## 9. Kompilasi PDF & Validasi Output

Mengompilasi berkas `.tex` menjadi `.pdf` menggunakan engine LaTeX lokal (**Tectonic**).
Hasil PDF diverifikasi keberadaannya dan diperiksa jumlah halamannya.


In [ ]:
compile_result = workflow.latex_engine.compile(
    job_mgr.tex_file,
    output_dir=job_mgr.job_dir,
    job_mgr=job_mgr
)

print("Status Kompilasi:")
print(f"  Sukses         : {compile_result['success']}")
print(f"  Exit Code      : {compile_result['exit_code']}")
if compile_result['success']:
    print(f"  Lokasi PDF     : {compile_result['pdf_path']}")
    print(f"  Jumlah Halaman : {compile_result['page_count']}")
    print(f"  Ukuran Berkas  : {compile_result['file_size']} bytes")
else:
    print(f"  Pesan Kesalahan: {compile_result.get('error')}")
    print(f"  Log Kompilasi  : {compile_result.get('log_path')}")


## 10. Ringkasan Eksekusi, Log & Pemulihan Pekerjaan

Menampilkan ringkasan lengkap metrik pekerjaan serta cara melanjutkan pekerjaan yang tertunda (*resumption*).


In [ ]:
import pprint

# Tampilkan ringkasan menyeluruh pekerjaan
summary = job_mgr.get_summary()
print("=" * 60)
print("RINGKASAN EKSEKUSI JOB")
print("=" * 60)
pprint.pprint(summary)
print("=" * 60)

# Menampilkan daftar pekerjaan yang tersimpan di disk
print("\nDaftar Pekerjaan Tersimpan di Direktori jobs/:")
all_saved_jobs = JobManager.list_jobs()
for j in all_saved_jobs:
    print(f"  Job ID: {j['job_id']:30} | Status: {j['status']:12} | Soal: {j['question_count']}")
